# Self-Consistency and Multiple Paths of Reasoning Tutorial

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/prompt-engineering/self-consistency.ipynb)

## Overview

This tutorial explores the concept of self-consistency and multiple paths of reasoning in prompt engineering. We'll focus on techniques for generating diverse reasoning paths and aggregating results to improve the quality and reliability of AI-generated answers.

## Motivation

Large language models can sometimes produce inconsistent or unreliable outputs. By leveraging multiple reasoning paths and aggregating results, we can enhance the robustness and accuracy of AI-generated responses. This approach is particularly useful for complex problem-solving tasks where a single path of reasoning might be insufficient or prone to errors.

## Key Components

1. Generating multiple reasoning paths
2. Aggregating results for better answers
3. Implementing self-consistency checks
4. Applying these techniques to various problem-solving scenarios

## Method Details

Our approach involves the following steps:

1. Setting up the environment with a local open-source model (Qwen3-8B via HuggingFace Transformers)
2. Designing prompts that encourage diverse reasoning paths
3. Generating multiple responses using sampling (temperature > 0) for diversity
4. Implementing aggregation methods (majority voting) to combine and analyze the generated responses
5. Applying self-consistency checks to evaluate the reliability of the results
6. Demonstrating the effectiveness of this approach on various problem types

Throughout the tutorial, we'll use practical examples to illustrate how these techniques can be applied to enhance the quality and reliability of AI-generated answers.

By the end of this tutorial, you'll have a solid understanding of how to implement self-consistency and multiple paths of reasoning in your prompt engineering workflows, leading to more robust and reliable AI-generated responses.

## Conclusion

This tutorial will equipped you with powerful techniques for enhancing the reliability and consistency of AI-generated responses through self-consistency and multiple paths of reasoning. By implementing these methods, you can:

1. Generate diverse problem-solving approaches, reducing the risk of biased or narrow solutions.
2. Aggregate multiple reasoning paths to arrive at more robust and reliable answers.
3. Apply self-consistency checks to evaluate and improve the quality of AI-generated outputs.
4. Adapt these techniques to various problem types, from factual queries to complex reasoning tasks.

Mastering these skills will significantly improve your ability to leverage AI language models for more accurate and trustworthy results across a wide range of applications. As you continue to explore and refine these techniques, you'll be better equipped to handle complex problems and generate high-quality, consistent outputs in your AI-driven projects.

## Why Multiple Samples Help: The Wisdom of Crowds

### Condorcet's Jury Theorem

The mathematical foundation for self-consistency comes from **Condorcet's Jury Theorem** (1785). The theorem states:

> If each independent voter (or "reasoner") has a probability $p > 0.5$ of being correct, then the probability that the **majority vote** is correct **increases monotonically** toward 1.0 as the number of voters increases.

Formally, for $n$ independent voters each with accuracy $p$, the probability that at least $\lceil n/2 \rceil$ are correct is:

$$P(\text{majority correct}) = \sum_{k=\lceil n/2 \rceil}^{n} \binom{n}{k} p^k (1-p)^{n-k}$$

This is exactly a **binomial CDF** calculation. Self-consistency leverages this: each sampled reasoning path acts as an independent "voter." If the model is more likely to be right than wrong on a given problem, then taking the majority answer across multiple samples will amplify that advantage.

The cell below simulates this mathematically — no LLM calls needed, just pure probability theory.

In [ ]:
import numpy as np
from scipy.stats import binom

# --- Condorcet's Jury Theorem Simulation ---
# For a model with individual accuracy p, compute majority-vote accuracy
# across different numbers of samples n.

individual_accuracies = [0.6, 0.7, 0.8]
sample_counts = [1, 3, 5, 7, 11, 15, 21]

print("Majority-Vote Accuracy (Condorcet's Jury Theorem)")
print("=" * 65)
print(f"{'Samples (n)':>12}", end="")
for p in individual_accuracies:
    print(f"  p={p:.1f}", end="")
print()
print("-" * 65)

for n in sample_counts:
    print(f"{n:>12}", end="")
    for p in individual_accuracies:
        # Majority = at least ceil(n/2) correct out of n
        majority_threshold = (n // 2) + 1
        # P(X >= threshold) = 1 - P(X < threshold) = 1 - CDF(threshold - 1)
        majority_acc = 1 - binom.cdf(majority_threshold - 1, n, p)
        print(f"  {majority_acc:>5.1%}", end="")
    print()

print("\n--- Key Insight ---")
print("Even a model that's only 60% accurate individually reaches ~74% with")
print("5 samples and ~84% with 21 samples. A 70% model hits ~97% at 21 samples.")
print("This is the mathematical engine behind self-consistency.")

### The KEY Caveat: LLM Errors Are NOT Independent

Condorcet's theorem requires **independence** between voters. In LLM self-consistency, this assumption is violated in important ways:

1. **Same model, same weights**: All samples come from the same neural network. If the model has a systematic misunderstanding (e.g., it learned an incorrect fact), temperature sampling won't fix it — every sample will reproduce the same error.

2. **Correlated errors**: At low temperatures, the model's outputs are highly correlated — it tends to follow the same high-probability reasoning path. Even at higher temperatures, the model's "error modes" are correlated because they stem from the same training distribution.

3. **Prompt sensitivity**: All samples receive the same prompt, so any ambiguity or bias in the prompt affects all samples equally.

**Bottom line**: Self-consistency amplifies the model's *stochastic* accuracy advantage — it reduces **random** errors caused by sampling noise. But it cannot overcome **systematic** errors where the model's knowledge or reasoning is fundamentally flawed. We'll demonstrate this distinction concretely in later sections.

## Setup

First, let's install dependencies and load a local open-source model using HuggingFace Transformers with 4-bit quantization on Google Colab.

In [ ]:
# --- Google Colab Setup ---
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import torch
from collections import Counter
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="auto", quantization_config=quantization_config,
    cache_dir=CACHE_DIR, torch_dtype="auto",
)

def generate(messages, max_new_tokens=1024, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs, max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None, do_sample=do_sample, top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

## Generating Multiple Reasoning Paths

Let's create functions that generate multiple diverse reasoning paths using sampling (temperature > 0) and aggregate them via majority voting.

In [ ]:
def generate_multiple_paths(messages, num_paths=5, temperature=0.8):
    """Generate multiple reasoning paths with sampling."""
    paths = []
    for i in range(num_paths):
        response = generate(messages, temperature=temperature, do_sample=True)
        paths.append(response)
        print(f"Path {i+1}: {response[:100]}...")
    return paths

def majority_vote(responses):
    """Extract final answers and return the most common one."""
    # Simple extraction: take the last line or sentence as the answer
    answers = []
    for r in responses:
        lines = r.strip().split('\n')
        answers.append(lines[-1].strip())
    vote = Counter(answers)
    print(f"\nVote distribution: {dict(vote)}")
    return vote.most_common(1)[0][0]

Now, let's test our function with a sample problem.

In [ ]:
problem = "A ball is thrown upwards with an initial velocity of 20 m/s. How high will it go?"

messages = [
    {"role": "system", "content": "You are a helpful assistant. Solve problems step by step."},
    {"role": "user", "content": f"Solve the following problem using a unique approach.\nProblem: {problem}"}
]

paths = generate_multiple_paths(messages, num_paths=3, temperature=0.8)

for i, path in enumerate(paths, 1):
    print(f"\nPath {i}:\n{path}\n")

## Temperature's Effect on Sample Diversity

Temperature controls the "sharpness" of the probability distribution over next tokens. Mathematically, for logits $z_i$:

$$P(\text{token}_i) = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}$$

- **T → 0**: Distribution collapses to argmax (greedy). All samples are nearly identical.
- **T = 1**: Standard softmax. Model's natural uncertainty is preserved.
- **T > 1**: Distribution flattens. More random, more diverse, but noisier.

For self-consistency, we need a **Goldilocks temperature**: high enough to get genuinely different reasoning paths, but low enough that each path is still "trying" to be correct. Let's see this empirically.

In [ ]:
# --- Temperature vs. Diversity Experiment ---
# Generate N=5 samples at different temperatures for the same problem.
# Measure: unique answers, agreement rate, and qualitative diversity.

import re

temp_test_problem = "What is 15% of 80?"
temp_test_messages = [
    {"role": "system", "content": "You are a helpful math tutor. Give your final numerical answer on the last line as: Answer: <number>"},
    {"role": "user", "content": f"Solve step by step: {temp_test_problem}"}
]

temperatures = [0.3, 0.6, 0.9, 1.2]
n_samples = 5

def extract_answer(text):
    """Try to extract a numerical answer from the response."""
    # Look for "Answer: X" pattern first
    match = re.search(r"[Aa]nswer:\s*([\d.]+)", text)
    if match:
        return match.group(1)
    # Fallback: last number in text
    numbers = re.findall(r"\b\d+\.?\d*\b", text)
    return numbers[-1] if numbers else text.strip().split("\n")[-1]

for temp in temperatures:
    print(f"\n{'=' * 60}")
    print(f"Temperature = {temp}")
    print(f"{'=' * 60}")
    answers = []
    for i in range(n_samples):
        response = generate(temp_test_messages, temperature=temp, do_sample=True, max_new_tokens=256)
        ans = extract_answer(response)
        answers.append(ans)
        print(f"  Sample {i+1}: Answer = {ans}")

    unique = set(answers)
    most_common = Counter(answers).most_common(1)[0]
    agreement = most_common[1] / len(answers)

    print(f"\n  Unique answers: {len(unique)} → {unique}")
    print(f"  Agreement rate:  {agreement:.0%} ({most_common[1]}/{len(answers)} agree on '{most_common[0]}')")

### Temperature Trade-off for Self-Consistency

| Temperature | Diversity | Each Sample Quality | Self-Consistency Benefit |
|:-----------:|:---------:|:-------------------:|:-----------------------:|
| 0.1–0.3 | Very low | High | **Minimal** — all samples say the same thing |
| 0.5–0.7 | Moderate | Good | **Optimal** — diverse paths, mostly correct |
| 0.8–1.0 | High | Acceptable | **Good** — more diversity, some noise |
| 1.2+ | Very high | Degraded | **Diminishing** — too much noise drowns signal |

**Recommendation**: For self-consistency, use temperatures in the **0.6–0.8 range**. This provides enough diversity to get genuinely different reasoning paths while keeping each individual path reasonably accurate. The original self-consistency paper (Wang et al., 2022) used T=0.7 as a default.

## Aggregating Results

Now that we have multiple reasoning paths, let's create a function to aggregate the results and determine the most consistent answer. We use two approaches: majority voting on final answers and an LLM-based aggregation.

In [ ]:
def aggregate_results(paths):
    """
    Aggregate results from multiple reasoning paths using the local model.

    Args:
        paths (list): List of reasoning paths.

    Returns:
        str: The most consistent answer.
    """
    joined_paths = "\n".join(paths)
    messages = [
        {"role": "system", "content": "You are a helpful assistant that synthesizes multiple reasoning paths."},
        {"role": "user", "content": (
            f"Analyze the following reasoning paths and determine the most consistent answer. "
            f"If there are discrepancies, explain why and provide the most likely correct answer.\n\n"
            f"Reasoning paths:\n{joined_paths}\n\nMost consistent answer:"
        )}
    ]
    return generate(messages, temperature=0.3, do_sample=True)

Let's apply both aggregation methods to our previous results.

In [ ]:
print("=== Majority Vote ===")
vote_result = majority_vote(paths)
print(f"Winner: {vote_result}")

print("\n=== LLM-based Aggregation ===")
aggregated_result = aggregate_results(paths)
print("Aggregated Result:\n", aggregated_result)

## Correlated Errors: When Majority Voting Fails

We established that self-consistency relies on errors being somewhat independent. Now let's see what happens when they're **not** — when the model makes the same mistake every time, regardless of sampling.

In [ ]:
# --- Demonstration: Systematic Error (All Samples Wrong) ---
# We'll ask a question where the model may have a common misconception.
# If the error is systematic, ALL samples will give the same wrong answer.

systematic_problem = (
    "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. "
    "How much does the ball cost? Think carefully and give your final answer as: Answer: $X.XX"
)

systematic_messages = [
    {"role": "system", "content": "Solve this problem step by step. Give your final answer on the last line as: Answer: $X.XX"},
    {"role": "user", "content": systematic_problem}
]

print("Testing for systematic errors with the classic bat-and-ball problem")
print("Correct answer: $0.05 (many humans and LLMs get this wrong as $0.10)\n")

n_trials = 7
answers_sys = []
for i in range(n_trials):
    resp = generate(systematic_messages, temperature=0.8, do_sample=True, max_new_tokens=512)
    ans = extract_answer(resp)
    answers_sys.append(ans)
    print(f"  Sample {i+1}: {ans}")

vote_dist = Counter(answers_sys)
print(f"\nVote distribution: {dict(vote_dist)}")
winner = vote_dist.most_common(1)[0][0]
print(f"Majority vote answer: {winner}")
print(f"\nIf all samples agree on the WRONG answer, more samples won't help.")
print("This is a correlated/systematic error — it's in the model's reasoning, not random noise.")

In [ ]:
# --- Does Temperature Fix Systematic Errors? ---
# Test the SAME problem at multiple temperatures to see if higher T helps.

print("Does varying temperature overcome systematic errors?")
print("=" * 60)

for temp in [0.3, 0.7, 1.0, 1.3]:
    temp_answers = []
    for i in range(5):
        resp = generate(systematic_messages, temperature=temp, do_sample=True, max_new_tokens=512)
        temp_answers.append(extract_answer(resp))
    dist = Counter(temp_answers)
    print(f"  T={temp:.1f}: {dict(dist)}")

print("\nObservation: If the model 'knows' the right answer, higher temperature")
print("won't change the systematic tendency. The error is in the model's weights,")
print("not in the sampling randomness.")

### When Does Self-Consistency Help vs. Not Help?

| Error Type | Example | Self-Consistency Helps? |
|:----------:|:-------:|:-----------------------:|
| **Random/Stochastic** | Model sometimes picks wrong arithmetic step | ✅ **Yes** — majority voting filters out random mistakes |
| **Near-threshold** | Model is ~55% accurate; some paths get it right | ✅ **Yes** — amplifies the slight accuracy advantage |
| **Systematic/Knowledge gap** | Model learned an incorrect fact | ❌ **No** — all samples reproduce the same error |
| **Prompt-induced bias** | Ambiguous prompt misleads all samples equally | ❌ **No** — bias is in the input, not the sampling |

**Key principle**: Self-consistency is a **variance reduction** technique, not a **bias correction** technique. It reduces random noise but cannot fix systematic biases. Understanding this distinction is essential for knowing when to apply (and when NOT to apply) self-consistency in production.

## Self-Consistency Check

To further improve our results, let's implement a self-consistency check that evaluates the reliability of our aggregated answer.

In [ ]:
def self_consistency_check(problem, aggregated_result):
    """
    Perform a self-consistency check on the aggregated result using the local model.

    Args:
        problem (str): The original problem statement.
        aggregated_result (str): The aggregated result to check.

    Returns:
        str: An evaluation of the result's consistency and reliability.
    """
    messages = [
        {"role": "system", "content": "You are a critical evaluator of reasoning quality."},
        {"role": "user", "content": (
            f"Evaluate the consistency and reliability of the following result for the given problem.\n\n"
            f"Problem: {problem}\n"
            f"Result: {aggregated_result}\n\n"
            f"Evaluation (consider factors like logical consistency, adherence to known facts, and potential biases):"
        )}
    ]
    return generate(messages, temperature=0.3, do_sample=True)

Now, let's apply the self-consistency check to our aggregated result.

In [ ]:
consistency_evaluation = self_consistency_check(problem, aggregated_result)
print("Self-Consistency Evaluation:\n", consistency_evaluation)

## Cost-Benefit Analysis: How Many Samples Are Enough?

In production, every additional sample means more compute, higher cost, and increased latency. Let's measure the trade-offs empirically and understand the **diminishing returns** curve.

In [ ]:
# --- Cost-Benefit: Token Usage and Accuracy vs. Number of Samples ---
# For a set of test problems, measure tokens per sample and track
# how majority-vote accuracy changes with increasing N.

import time

cost_problem = "What is the sum of the first 20 positive integers?"
correct_answer = "210"  # n(n+1)/2 = 20*21/2 = 210

cost_messages = [
    {"role": "system", "content": "Solve the math problem step by step. Put your final numerical answer on the last line as: Answer: <number>"},
    {"role": "user", "content": f"Problem: {cost_problem}"}
]

# Generate a larger pool of samples and measure tokens
max_samples = 11
all_responses = []
all_answers = []
token_counts = []

print(f"Generating {max_samples} samples for cost analysis...")
print(f"Problem: {cost_problem}")
print(f"Correct answer: {correct_answer}\n")

for i in range(max_samples):
    start = time.time()
    resp = generate(cost_messages, temperature=0.7, do_sample=True, max_new_tokens=512)
    elapsed = time.time() - start
    ans = extract_answer(resp)
    n_tokens = len(tokenizer.encode(resp))
    all_responses.append(resp)
    all_answers.append(ans)
    token_counts.append(n_tokens)
    print(f"  Sample {i+1}: answer={ans}, tokens={n_tokens}, time={elapsed:.1f}s")

# Now simulate majority voting at different N values
print(f"\n{'=' * 60}")
print("Cost-Benefit Analysis")
print(f"{'=' * 60}")
print(f"{'N samples':>10} {'Total tokens':>14} {'Majority answer':>16} {'Correct?':>10} {'Marginal gain':>14}")
print("-" * 70)

prev_correct = False
for n in [1, 3, 5, 7, 9, 11]:
    if n > len(all_answers):
        break
    subset = all_answers[:n]
    total_tokens = sum(token_counts[:n])
    vote = Counter(subset).most_common(1)[0][0]
    is_correct = (vote == correct_answer)
    marginal = "—" if n == 1 else ("+✓" if is_correct and not prev_correct else ("same" if is_correct == prev_correct else "-✗"))
    prev_correct = is_correct
    print(f"{n:>10} {total_tokens:>14,} {vote:>16} {'✓' if is_correct else '✗':>10} {marginal:>14}")

avg_tokens = np.mean(token_counts)
print(f"\nAverage tokens per sample: {avg_tokens:.0f}")
print(f"Cost scales linearly: {max_samples} samples ≈ {max_samples}x the cost of 1 sample")

### Diminishing Returns: The Practical Sweet Spot

The relationship between number of samples and accuracy follows a pattern of **diminishing marginal returns**:

| Transition | Accuracy Gain | Cost Increase | Worth It? |
|:----------:|:-------------:|:-------------:|:---------:|
| 1 → 3 | **Large** | 3× | ✅ Almost always |
| 3 → 5 | Moderate | 1.67× | ✅ Usually |
| 5 → 7 | Small | 1.4× | ⚠️ Depends on task criticality |
| 7 → 11 | Very small | 1.57× | ❌ Rarely justified |
| 11 → 21 | Negligible | 1.91× | ❌ Almost never |

**Production recommendations**:
- **Low-stakes tasks** (chatbot, summarization): N=1 is fine. Self-consistency is overkill.
- **Medium-stakes** (code generation, data extraction): N=3–5. Best bang for the buck.
- **High-stakes** (medical, legal, financial): N=5–7. The extra cost is justified by reliability.
- **Critical decisions**: N=7+, but also consider combining with other techniques (chain-of-thought, tool use, human review).

**Latency note**: If you can parallelize generation (batch inference), N samples only increase *cost*, not *latency*. If sequential, latency scales linearly — a crucial factor for real-time applications.

## Applying to Different Problem Types

Let's demonstrate how this approach can be applied to different types of problems.

In [ ]:
def solve_problem(problem):
    """
    Solve a problem using multiple reasoning paths, aggregation, and self-consistency check.

    Args:
        problem (str): The problem statement.

    Returns:
        tuple: (aggregated_result, consistency_evaluation)
    """
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Solve problems step by step."},
        {"role": "user", "content": f"Solve the following problem using a unique approach.\nProblem: {problem}"}
    ]
    paths = generate_multiple_paths(messages, num_paths=3, temperature=0.8)
    vote_result = majority_vote(paths)
    print(f"Majority vote winner: {vote_result}")
    aggregated_result = aggregate_results(paths)
    consistency_evaluation = self_consistency_check(problem, aggregated_result)
    return aggregated_result, consistency_evaluation

# Example problems
problems = [
    "What is the capital of France?",
    "Explain the concept of supply and demand in economics.",
    "If a train travels at 60 km/h, how long will it take to cover 180 km?"
]

for problem in problems:
    print(f"Problem: {problem}")
    result, evaluation = solve_problem(problem)
    print("\nAggregated Result:\n", result)
    print("\nConsistency Evaluation:\n", evaluation)
    print("\n" + "-"*50 + "\n")

## Advanced Voting Strategies

Simple majority voting treats all samples equally. But not all reasoning paths are created equal — some are more confident, more detailed, or more internally consistent. Let's explore weighted and filtered voting strategies.

In [ ]:
# --- Advanced Voting: Majority vs. Weighted vs. Unanimous ---
# Compare three voting strategies on a reasoning problem.

voting_problem = "A farmer has 17 sheep. All but 9 die. How many are left?"
voting_correct = "9"

voting_messages = [
    {"role": "system", "content": "Solve the problem step by step. End with: Answer: <number>"},
    {"role": "user", "content": f"Problem: {voting_problem}"}
]

# Generate samples with token-level log probabilities (approximated by length/confidence)
n_vote_samples = 7
vote_responses = []
vote_answers = []
vote_lengths = []  # Proxy for "confidence" — shorter, more direct answers may be more confident

print(f"Problem: {voting_problem}")
print(f"Correct answer: {voting_correct}\n")

for i in range(n_vote_samples):
    resp = generate(voting_messages, temperature=0.8, do_sample=True, max_new_tokens=256)
    ans = extract_answer(resp)
    vote_responses.append(resp)
    vote_answers.append(ans)
    vote_lengths.append(len(resp))
    print(f"  Sample {i+1}: answer={ans}, length={len(resp)} chars")

# --- Strategy 1: Simple Majority Vote ---
simple_vote = Counter(vote_answers).most_common(1)[0]
print(f"\n--- Strategy 1: Simple Majority Vote ---")
print(f"  Distribution: {dict(Counter(vote_answers))}")
print(f"  Winner: {simple_vote[0]} ({simple_vote[1]}/{n_vote_samples} votes)")
print(f"  Correct? {'✓' if simple_vote[0] == voting_correct else '✗'}")

# --- Strategy 2: Weighted Vote (inversely weighted by length — shorter = more confident) ---
print(f"\n--- Strategy 2: Weighted Vote (shorter = higher weight) ---")
max_len = max(vote_lengths)
weights = [max_len / l for l in vote_lengths]  # Inverse length as confidence proxy
weighted_scores = {}
for ans, w in zip(vote_answers, weights):
    weighted_scores[ans] = weighted_scores.get(ans, 0) + w
weighted_winner = max(weighted_scores, key=weighted_scores.get)
for ans, score in sorted(weighted_scores.items(), key=lambda x: -x[1]):
    print(f"  {ans}: weighted score = {score:.2f}")
print(f"  Winner: {weighted_winner}")
print(f"  Correct? {'✓' if weighted_winner == voting_correct else '✗'}")

# --- Strategy 3: Unanimous Agreement ---
print(f"\n--- Strategy 3: Unanimous Agreement ---")
unique_answers = set(vote_answers)
if len(unique_answers) == 1:
    unanimous_ans = unique_answers.pop()
    print(f"  All {n_vote_samples} samples agree: {unanimous_ans}")
    print(f"  HIGH CONFIDENCE — Correct? {'✓' if unanimous_ans == voting_correct else '✗'}")
else:
    print(f"  No unanimity — {len(unique_answers)} distinct answers: {unique_answers}")
    print(f"  Action: Fall back to majority vote or flag for human review")
    print(f"  Majority fallback: {simple_vote[0]}")

### When to Use Each Voting Strategy

| Strategy | How It Works | Best For | Risk |
|:--------:|:------------:|:--------:|:----:|
| **Simple Majority** | Each sample = 1 vote | General-purpose, most tasks | Treats confident and uncertain samples equally |
| **Weighted Vote** | Weight by confidence/probability | When you have access to log-probs or confidence signals | Weight heuristic may be noisy |
| **Unanimous** | Accept only if ALL agree | High-stakes decisions; use as a confidence filter | Very conservative — often no answer |
| **Threshold** | Accept if ≥X% agree | Tunable confidence level | Requires calibrating the threshold |

**Practical guidance**:
- Start with **simple majority** — it's robust and well-studied.
- Use **unanimous agreement** as a **confidence signal**, not a voting method: if all 5 samples agree, you can be more confident; if they disagree, flag for review.
- **Weighted voting** shines when you can get actual generation probabilities (log-probs from the API). Using response length as a proxy is a rough heuristic.
- In production, combine voting with a **confidence threshold**: "accept the majority answer only if ≥60% of samples agree; otherwise, escalate to human review."